In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/blastchar/improve-customer-retention-churn/Rplot001.png
/kaggle/input/notebooks/blastchar/improve-customer-retention-churn/__results__.html
/kaggle/input/notebooks/blastchar/improve-customer-retention-churn/__output__.json
/kaggle/input/notebooks/blastchar/improve-customer-retention-churn/custom.css
/kaggle/input/notebooks/blastchar/improve-customer-retention-churn/__results___files/__results___28_1.png
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv
/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
 
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from category_encoders import TargetEncoder
 
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
 
SEED     = 42
N_SPLITS = 5
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

 

In [3]:
print("SECTION 2 │ Loading Data")
 
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")
 
TARGET = "Churn"
ID_COL = "id"
 
# Normalise target
train[TARGET] = train[TARGET].map({"Yes": 1, "No": 0, 1: 1, 0: 0, "1": 1, "0": 0}).astype(int)
 
print(f"Train : {train.shape}  |  Test : {test.shape}")
 
try:
    orig = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
    orig[TARGET] = orig[TARGET].map({"Yes": 1, "No": 0}).astype(int)
    # Drop customerID, align columns to train
    orig = orig.drop(columns=["customerID"], errors="ignore")
    # Only keep rows + cols that exist in train
    shared_cols = [c for c in orig.columns if c in train.columns]
    orig = orig[shared_cols]
    # Give dummy ids so concat works
    orig[ID_COL] = -1
    train = pd.concat([train, orig], ignore_index=True)
    print(f"Original Telco data merged — new train size: {train.shape[0]:,}")
except FileNotFoundError:
    print("Original Telco CSV not found — skipping merge (download from Kaggle for +AUC boost)")
 
churn_counts     = train[TARGET].value_counts().sort_index()
imbalance_ratio  = churn_counts[0] / churn_counts[1]
USE_CLASS_WEIGHT = imbalance_ratio > 3
scale_pos        = int(imbalance_ratio) if USE_CLASS_WEIGHT else 1
print(f"Churn rate : {train[TARGET].mean():.3f}  |  Imbalance : {imbalance_ratio:.2f}:1")
 
num_cols = [c for c in train.select_dtypes(include=[np.number]).columns if c not in [ID_COL, TARGET]]
cat_cols = [c for c in train.select_dtypes(include="object").columns  if c not in [ID_COL, TARGET]]
 

SECTION 2 │ Loading Data
Train : (594194, 21)  |  Test : (254655, 20)
Original Telco data merged — new train size: 601,237
Churn rate : 0.226  |  Imbalance : 3.43:1


FEATURE ENGINEERING

In [4]:
print("SECTION 3 │ Feature Engineering")

 
def engineer_features(df, ref_df=None):
    df  = df.copy()
    ref = ref_df if ref_df is not None else df
 
    if "TotalCharges" in df.columns:
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
 
    # Tenure
    if "tenure" in df.columns:
        df["tenure_group"]  = pd.cut(df["tenure"], bins=[0,6,12,24,48,60,np.inf],
                                      labels=[1,2,3,4,5,6]).astype(float)
        df["is_new"]        = (df["tenure"] <= 6).astype(int)
        df["is_mid"]        = ((df["tenure"] > 6) & (df["tenure"] <= 24)).astype(int)
        df["is_long"]       = (df["tenure"] >= 48).astype(int)
        df["log_tenure"]    = np.log1p(df["tenure"])
        df["tenure_inv"]    = 1 / (df["tenure"] + 1)
        df["tenure_sq"]     = df["tenure"] ** 2
 
    # Monthly charges
    if "MonthlyCharges" in df.columns:
        mc_mean = ref["MonthlyCharges"].mean()
        mc_std  = ref["MonthlyCharges"].std()
        df["log_monthly"]     = np.log1p(df["MonthlyCharges"])
        df["monthly_zscore"]  = (df["MonthlyCharges"] - mc_mean) / (mc_std + 1e-6)
        df["monthly_vs_mean"] = df["MonthlyCharges"] / (mc_mean + 1e-6)
        df["is_high_value"]   = (df["MonthlyCharges"] > ref["MonthlyCharges"].quantile(0.75)).astype(int)
        df["is_low_value"]    = (df["MonthlyCharges"] < ref["MonthlyCharges"].quantile(0.25)).astype(int)
 
    # Tenure × charge
    if "MonthlyCharges" in df.columns and "tenure" in df.columns:
        df["charges_per_tenure"]  = df["MonthlyCharges"] / (df["tenure"] + 1)
        df["tenure_x_monthly"]    = df["tenure"] * df["MonthlyCharges"]
        df["log_t_x_log_m"]       = np.log1p(df["tenure"]) * np.log1p(df["MonthlyCharges"])
        df["monthly_per_log_t"]   = df["MonthlyCharges"] / (np.log1p(df["tenure"]) + 1)
 
    # Total charges
    if all(c in df.columns for c in ["TotalCharges", "MonthlyCharges", "tenure"]):
        df["expected_total"]       = df["MonthlyCharges"] * df["tenure"]
        df["charge_deviation"]     = df["TotalCharges"] - df["expected_total"]
        df["charge_deviation_pct"] = df["charge_deviation"] / (df["expected_total"] + 1)
        df["log_total"]            = np.log1p(df["TotalCharges"])
        df["total_to_monthly"]     = df["TotalCharges"] / (df["MonthlyCharges"] + 1)
        df["total_to_expected"]    = df["TotalCharges"] / (df["expected_total"] + 1)
 
    # Services
    svc = [c for c in ["PhoneService","MultipleLines","InternetService",
                        "OnlineSecurity","OnlineBackup","DeviceProtection",
                        "TechSupport","StreamingTV","StreamingMovies"] if c in df.columns]
    if svc:
        df["num_services"]    = (df[svc] == "Yes").sum(axis=1)
        df["has_no_services"] = (df["num_services"] == 0).astype(int)
        df["has_streaming"]   = (
            (df.get("StreamingTV",     pd.Series("No", index=df.index)) == "Yes") |
            (df.get("StreamingMovies", pd.Series("No", index=df.index)) == "Yes")
        ).astype(int)
        if "MonthlyCharges" in df.columns:
            df["charge_per_service"] = df["MonthlyCharges"] / (df["num_services"] + 1)
 
    # Contract & payment
    if "Contract" in df.columns:
        df["contract_risk"]       = df["Contract"].map(
            {"Month-to-month": 2, "One year": 1, "Two year": 0}).fillna(1)
        df["is_monthly_contract"] = (df["Contract"] == "Month-to-month").astype(int)
 
    if "PaymentMethod" in df.columns:
        df["auto_pay"]      = df["PaymentMethod"].str.contains("automatic", case=False, na=False).astype(int)
        df["is_electronic"] = df["PaymentMethod"].str.contains("electronic", case=False, na=False).astype(int)
 
    # Composite risk score
    risk = pd.Series(0.0, index=df.index)
    if "contract_risk"  in df.columns: risk += df["contract_risk"]  * 0.35
    if "auto_pay"       in df.columns: risk += (1 - df["auto_pay"]) * 0.20
    if "is_new"         in df.columns: risk += df["is_new"]         * 0.20
    if "num_services"   in df.columns: risk += (1 / (df["num_services"] + 1)) * 0.15
    if "is_high_value"  in df.columns: risk += df["is_high_value"]  * 0.10
    df["composite_risk"] = risk
 
    # Interactions
    if "contract_risk" in df.columns:
        if "num_services"       in df.columns: df["risk_x_services"] = df["contract_risk"] * df["num_services"]
        if "charges_per_tenure" in df.columns: df["risk_x_charge"]   = df["contract_risk"] * df["charges_per_tenure"]
        if "tenure_inv"         in df.columns: df["risk_x_new"]      = df["contract_risk"] * df["tenure_inv"]
    if "composite_risk" in df.columns and "MonthlyCharges" in df.columns:
        df["risk_x_monthly"] = df["composite_risk"] * df["MonthlyCharges"]
    if "SeniorCitizen" in df.columns:
        if "MonthlyCharges" in df.columns: df["senior_x_monthly"] = df["SeniorCitizen"] * df["MonthlyCharges"]
        if "contract_risk"  in df.columns: df["senior_x_risk"]    = df["SeniorCitizen"] * df["contract_risk"]
 
    return df
 
train_raw = train.copy()
train = engineer_features(train, ref_df=train_raw)
test  = engineer_features(test,  ref_df=train_raw)
print(f"Train shape after engineering : {train.shape}")
 
all_cat = [c for c in train.select_dtypes(include="object").columns if c not in [ID_COL, TARGET]]

SECTION 3 │ Feature Engineering
Train shape after engineering : (601237, 58)


SECTION 4 │ TWO FEATURE VIEWS

Boosting models  → target encoded   (strong signal, ok with leakage fix)

RF / ET          → label encoded    (different view of the same data)

In [5]:
# View A: Label encoded (for RF/ET and base reference) 
train_le = train.copy()
test_le  = test.copy()
le_dict  = {}
for col in all_cat:
    le = LabelEncoder()
    combined = pd.concat([train_le[col], test_le[col]], axis=0).astype(str)
    le.fit(combined)
    train_le[col] = le.transform(train_le[col].astype(str))
    test_le[col]  = le.transform(test_le[col].astype(str))
    le_dict[col]  = le
 
train_le.fillna(train_le.median(numeric_only=True), inplace=True)
test_le.fillna(train_le.median(numeric_only=True),  inplace=True)
 
feat_cols = [c for c in train_le.columns if c not in [ID_COL, TARGET]]
X_le      = train_le[feat_cols].reset_index(drop=True)
X_test_le = test_le[feat_cols].reset_index(drop=True)
y         = train_le[TARGET].reset_index(drop=True).astype(int)
 
print(f"View A (label-encoded)  features : {X_le.shape[1]}")
print(f"View B (target-encoded) — computed per fold inside CV loop")
 
 

View A (label-encoded)  features : 56
View B (target-encoded) — computed per fold inside CV loop


SECTION 5 │ 5-FOLD CV

XGB / LGB / CAT  trained on target-encoded features (per fold)

 RF  / ET          trained on label-encoded features

In [6]:
models   = ["XGB", "LGB", "CAT", "RF", "ET"]
oof      = {m: np.zeros(len(X_le))      for m in models}
test_p   = {m: np.zeros(len(X_test_le)) for m in models}
fold_auc = {m: []                        for m in models}
 
# Store raw train/test for target encoding inside fold
train_te_base = train[[ID_COL, TARGET] + list(feat_cols)].copy()
 
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_le, y), 1):
    print(f"\n  ── Fold {fold}/{N_SPLITS} ──")
 
    # Label-encoded splits (for RF / ET) 
    X_tr_le,  X_val_le  = X_le.iloc[tr_idx].copy(),  X_le.iloc[val_idx].copy()
    y_tr,     y_val     = y.iloc[tr_idx],              y.iloc[val_idx]
 
    #  Target-encoded splits (for XGB / LGB / CAT) 
    X_tr_raw  = train[feat_cols].iloc[tr_idx].copy()
    X_val_raw = train[feat_cols].iloc[val_idx].copy()
    X_test_raw = test[feat_cols].copy()
 
    # Fit target encoder on train fold only
    te = TargetEncoder(cols=all_cat, smoothing=10)
    X_tr_raw[all_cat]  = te.fit_transform(X_tr_raw[all_cat], y_tr)
    X_val_raw[all_cat] = te.transform(X_val_raw[all_cat])
    X_test_raw[all_cat] = te.transform(X_test_raw[all_cat])
 
    X_tr_raw.fillna(X_tr_raw.median(), inplace=True)
    X_val_raw.fillna(X_tr_raw.median(), inplace=True)
    X_test_raw.fillna(X_tr_raw.median(), inplace=True)
 
    # XGBoost — target encoded features
    m = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=scale_pos,
        eval_metric="auc", early_stopping_rounds=50,
        random_state=SEED, n_jobs=-1, verbosity=0
    )
    m.fit(X_tr_raw, y_tr, eval_set=[(X_val_raw, y_val)], verbose=False)
    oof["XGB"][val_idx] = m.predict_proba(X_val_raw)[:, 1]
    test_p["XGB"]      += m.predict_proba(X_test_raw)[:, 1] / N_SPLITS
    s = roc_auc_score(y_val, oof["XGB"][val_idx])
    fold_auc["XGB"].append(s); print(f"     XGB  AUC: {s:.4f}")
 
    # LightGBM — target encoded features 
    m = lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        reg_alpha=0.1, reg_lambda=1.0,
        class_weight="balanced" if USE_CLASS_WEIGHT else None,
        random_state=SEED, n_jobs=-1, verbose=-1
    )
    m.fit(X_tr_raw, y_tr, eval_set=[(X_val_raw, y_val)],
          callbacks=[lgb.early_stopping(50, verbose=False),
                     lgb.log_evaluation(period=-1)])
    oof["LGB"][val_idx] = m.predict_proba(X_val_raw)[:, 1]
    test_p["LGB"]      += m.predict_proba(X_test_raw)[:, 1] / N_SPLITS
    s = roc_auc_score(y_val, oof["LGB"][val_idx])
    fold_auc["LGB"].append(s); print(f"     LGB  AUC: {s:.4f}")
 
    # CatBoost — target encoded features 
    m = CatBoostClassifier(
        iterations=1000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, bagging_temperature=0.5,
        auto_class_weights="Balanced" if USE_CLASS_WEIGHT else None,
        eval_metric="AUC", early_stopping_rounds=50,
        random_seed=SEED, verbose=0
    )
    m.fit(X_tr_raw, y_tr, eval_set=(X_val_raw, y_val))
    oof["CAT"][val_idx] = m.predict_proba(X_val_raw)[:, 1]
    test_p["CAT"]      += m.predict_proba(X_test_raw)[:, 1] / N_SPLITS
    s = roc_auc_score(y_val, oof["CAT"][val_idx])
    fold_auc["CAT"].append(s); print(f"     CAT  AUC: {s:.4f}")
 
    #RandomForest — label encoded features 
    m = RandomForestClassifier(
        n_estimators=600, max_depth=14, min_samples_leaf=3,
        max_features="sqrt", max_samples=0.8,
        class_weight="balanced" if USE_CLASS_WEIGHT else None,
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_tr_le, y_tr)
    oof["RF"][val_idx] = m.predict_proba(X_val_le)[:, 1]
    test_p["RF"]      += m.predict_proba(X_test_le)[:, 1] / N_SPLITS
    s = roc_auc_score(y_val, oof["RF"][val_idx])
    fold_auc["RF"].append(s); print(f"     RF   AUC: {s:.4f}")
 
    # ExtraTrees — label encoded features 
    m = ExtraTreesClassifier(
        n_estimators=600, max_depth=14, min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced" if USE_CLASS_WEIGHT else None,
        random_state=SEED, n_jobs=-1
    )
    m.fit(X_tr_le, y_tr)
    oof["ET"][val_idx] = m.predict_proba(X_val_le)[:, 1]
    test_p["ET"]      += m.predict_proba(X_test_le)[:, 1] / N_SPLITS
    s = roc_auc_score(y_val, oof["ET"][val_idx])
    fold_auc["ET"].append(s); print(f"     ET   AUC: {s:.4f}")
 
# Summary
print("\n  OOF Summary ")
for m_name in models:
    sc = fold_auc[m_name]
    print(f"  {m_name:5s}  mean={np.mean(sc):.4f}  std={np.std(sc):.4f}")
 
 


  ── Fold 1/5 ──
     XGB  AUC: 0.9156
     LGB  AUC: 0.9149
     CAT  AUC: 0.9156
     RF   AUC: 0.9131
     ET   AUC: 0.9128

  ── Fold 2/5 ──
     XGB  AUC: 0.9163
     LGB  AUC: 0.9155
     CAT  AUC: 0.9160
     RF   AUC: 0.9133
     ET   AUC: 0.9128

  ── Fold 3/5 ──
     XGB  AUC: 0.9139
     LGB  AUC: 0.9134
     CAT  AUC: 0.9137
     RF   AUC: 0.9115
     ET   AUC: 0.9109

  ── Fold 4/5 ──
     XGB  AUC: 0.9149
     LGB  AUC: 0.9142
     CAT  AUC: 0.9150
     RF   AUC: 0.9123
     ET   AUC: 0.9118

  ── Fold 5/5 ──
     XGB  AUC: 0.9159
     LGB  AUC: 0.9155
     CAT  AUC: 0.9159
     RF   AUC: 0.9131
     ET   AUC: 0.9127

  OOF Summary 
  XGB    mean=0.9153  std=0.0008
  LGB    mean=0.9147  std=0.0008
  CAT    mean=0.9152  std=0.0008
  RF     mean=0.9127  std=0.0006
  ET     mean=0.9122  std=0.0007


PSEUDO-LABELLING

In [7]:
def rank_avg(preds_dict):
    n  = len(list(preds_dict.values())[0])
    ra = np.zeros(n)
    for p in preds_dict.values():
        ra += rankdata(p) / n
    return ra / len(preds_dict)
 
test_base   = rank_avg(test_p)
CONF_LOW    = 0.05
CONF_HIGH   = 0.95
pseudo_mask = (test_base < CONF_LOW) | (test_base > CONF_HIGH)
pseudo_y    = (test_base[pseudo_mask] > 0.5).astype(int)
pseudo_X    = X_le[pseudo_mask].copy() if False else X_test_le[pseudo_mask].copy()
pseudo_ys   = pd.Series(pseudo_y, index=pseudo_X.index)
 
print(f"Selected : {pseudo_mask.sum():,} / {len(test_base):,} rows")
print(f"  label=1 : {(pseudo_y==1).sum():,}  label=0 : {(pseudo_y==0).sum():,}")
 
X_aug_le = pd.concat([X_le,  pseudo_X],  ignore_index=True)
y_aug    = pd.concat([y,     pseudo_ys], ignore_index=True)
 
# Re-train on augmented label-encoded data (RF + ET)
print("  Re-training RF / ET on augmented data...")
rf_aug = RandomForestClassifier(
    n_estimators=600, max_depth=14, min_samples_leaf=3,
    max_features="sqrt", max_samples=0.8,
    class_weight="balanced" if USE_CLASS_WEIGHT else None,
    random_state=SEED, n_jobs=-1
)
rf_aug.fit(X_aug_le, y_aug)
 
et_aug = ExtraTreesClassifier(
    n_estimators=600, max_depth=14, min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced" if USE_CLASS_WEIGHT else None,
    random_state=SEED, n_jobs=-1
)
et_aug.fit(X_aug_le, y_aug)
 
# Also re-train XGB/LGB/CAT on augmented target-encoded data
# Use target encoding fit on full train (acceptable for augmented re-train)
te_full = TargetEncoder(cols=all_cat, smoothing=10)
X_train_te = train[feat_cols].copy()
X_train_te[all_cat] = te_full.fit_transform(X_train_te[all_cat], y)
X_train_te.fillna(X_train_te.median(), inplace=True)
 
X_test_te_full = test[feat_cols].copy()
X_test_te_full[all_cat] = te_full.transform(X_test_te_full[all_cat])
X_test_te_full.fillna(X_train_te.median(), inplace=True)
 
pseudo_X_te = X_test_te_full[pseudo_mask].copy()
X_aug_te    = pd.concat([X_train_te, pseudo_X_te], ignore_index=True)
 
print("  Re-training XGB / LGB / CAT on augmented data...")
xgb_aug = xgb.XGBClassifier(
    n_estimators=1000, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    scale_pos_weight=scale_pos, random_state=SEED, n_jobs=-1, verbosity=0
)
xgb_aug.fit(X_aug_te, y_aug, verbose=False)
 
lgb_aug = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced" if USE_CLASS_WEIGHT else None,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_aug.fit(X_aug_te, y_aug)
 
cat_aug = CatBoostClassifier(
    iterations=1000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
    auto_class_weights="Balanced" if USE_CLASS_WEIGHT else None,
    eval_metric="AUC", random_seed=SEED, verbose=0
)
cat_aug.fit(X_aug_te, y_aug)
 
# Blend: 60% CV + 40% augmented
test_p["XGB"] = 0.6*test_p["XGB"] + 0.4*xgb_aug.predict_proba(X_test_te_full)[:, 1]
test_p["LGB"] = 0.6*test_p["LGB"] + 0.4*lgb_aug.predict_proba(X_test_te_full)[:, 1]
test_p["CAT"] = 0.6*test_p["CAT"] + 0.4*cat_aug.predict_proba(X_test_te_full)[:, 1]
test_p["RF"]  = 0.6*test_p["RF"]  + 0.4*rf_aug.predict_proba(X_test_le)[:, 1]
test_p["ET"]  = 0.6*test_p["ET"]  + 0.4*et_aug.predict_proba(X_test_le)[:, 1]
 
 

Selected : 23,173 / 254,655 rows
  label=1 : 12,438  label=0 : 10,735
  Re-training RF / ET on augmented data...
  Re-training XGB / LGB / CAT on augmented data...


STACKING META-LEARNER

In [8]:
oof_stack  = np.column_stack([oof[m]    for m in models])
test_stack = np.column_stack([test_p[m] for m in models])
 
scaler     = StandardScaler()
oof_s      = scaler.fit_transform(oof_stack)
test_s     = scaler.transform(test_stack)
 
meta_lr_oof  = np.zeros(len(X_le)); meta_lr_test  = np.zeros(len(X_test_le))
meta_rdg_oof = np.zeros(len(X_le)); meta_rdg_test = np.zeros(len(X_test_le))
 
for tr_i, val_i in StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                                     random_state=SEED).split(oof_s, y):
    lr = LogisticRegression(C=0.5, max_iter=2000, random_state=SEED)
    lr.fit(oof_s[tr_i], y.iloc[tr_i])
    meta_lr_oof[val_i]  = lr.predict_proba(oof_s[val_i])[:, 1]
    meta_lr_test       += lr.predict_proba(test_s)[:, 1] / N_SPLITS
 
    rdg = Ridge(alpha=1.0)
    rdg.fit(oof_s[tr_i], y.iloc[tr_i])
    meta_rdg_oof[val_i] = rdg.predict(oof_s[val_i])
    meta_rdg_test      += rdg.predict(test_s) / N_SPLITS
 
meta_rdg_oof  = np.clip(meta_rdg_oof,  0, 1)
meta_rdg_test = np.clip(meta_rdg_test, 0, 1)
meta_oof  = 0.5*meta_lr_oof  + 0.5*meta_rdg_oof
meta_test = 0.5*meta_lr_test + 0.5*meta_rdg_test
 
print(f"  LR  meta OOF AUC : {roc_auc_score(y, meta_lr_oof):.4f}")
print(f"  RDG meta OOF AUC : {roc_auc_score(y, meta_rdg_oof):.4f}")
print(f"  AVG meta OOF AUC : {roc_auc_score(y, meta_oof):.4f}")
 
 

  LR  meta OOF AUC : 0.9155
  RDG meta OOF AUC : 0.9125
  AVG meta OOF AUC : 0.9150


FINAL BLEND

In [9]:
base_aucs = {m: roc_auc_score(y, oof[m]) for m in models}
total_auc = sum(base_aucs.values())
weights   = {m: base_aucs[m] / total_auc for m in models}
 
print("  Per-model OOF AUC & weight:")
for m in models:
    print(f"    {m:5s}  AUC={base_aucs[m]:.4f}  w={weights[m]:.3f}")
 
n  = len(X_le); nt = len(X_test_le)
blend_oof  = sum(weights[m] * rankdata(oof[m])    / n  for m in models)
blend_test = sum(weights[m] * rankdata(test_p[m]) / nt for m in models)
 
auc_blend = roc_auc_score(y, blend_oof)
print(f"\n  Weighted rank-blend OOF AUC : {auc_blend:.4f}")
 
final_oof  = 0.5*meta_oof  + 0.5*blend_oof
final_test = 0.5*meta_test + 0.5*blend_test
auc_final  = roc_auc_score(y, final_oof)
print(f"  FINAL combined OOF AUC     : {auc_final:.4f}")
 

  Per-model OOF AUC & weight:
    XGB    AUC=0.9153  w=0.200
    LGB    AUC=0.9147  w=0.200
    CAT    AUC=0.9152  w=0.200
    RF     AUC=0.9126  w=0.200
    ET     AUC=0.9122  w=0.200

  Weighted rank-blend OOF AUC : 0.9150
  FINAL combined OOF AUC     : 0.9153


SUBMISSION

In [10]:
submission = pd.DataFrame({"id": test["id"], "Churn": final_test})
submission.to_csv("submission.csv", index=False)
 
print(f"  Shape : {submission.shape}")
print(f"  Mean  : {final_test.mean():.4f}  |  Std : {final_test.std():.4f}")
print(f"  → submission.csv saved")
print(f"""

Previous best   :  0.9161                           
Stack OOF AUC   :  {roc_auc_score(y, meta_oof):.4f}                         
Blend OOF AUC   :  {auc_blend:.4f}                         
FINAL OOF AUC   :  {auc_final:.4f}                         
""")

  Shape : (254655, 2)
  Mean  : 0.3616  |  Std : 0.2717
  → submission.csv saved


Previous best   :  0.9161                           
Stack OOF AUC   :  0.9150                         
Blend OOF AUC   :  0.9150                         
FINAL OOF AUC   :  0.9153                         

